# Глава 6. Тонкая настройка для классификации текста

In [1]:
pip install matplotlib numpy tiktoken torch tensorflow pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from importlib.metadata import version

pkgs = ["matplotlib",  # Библиотека для построения графиков
        "numpy",       # Зависимость для PyTorch и TensorFlow
        "tiktoken",    # Токенизатор
        "torch",       # Библиотека для глубокого обучения
        "tensorflow",  # Для предобученных весов OpenAI
        "pandas"       # Загрузка наборов данных
       ]
for p in pkgs:
    print(f"{p} версия: {version(p)}")

matplotlib версия: 3.10.9
numpy версия: 2.4.4
tiktoken версия: 0.12.0
torch версия: 2.12.0
tensorflow версия: 2.21.0
pandas версия: 3.0.3


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/01.webp" width=800px>

&nbsp;
### 6.1. Различные категории тонкой настройки

- Наиболее распространёнными способами тонкой настройки языковых моделей являются инструктивная тонкая настройка и тонкая настройка для классификации
- Инструктивная тонкая настройка, изображённая ниже, является темой следующей главы

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/02.webp" width=800px>

- Тонкая настройка для классификации — тема этой главы. Это процедура, с которой вы, возможно, уже знакомы, если имеете опыт в машинном обучении: она похожа, например, на обучение свёрточной сети распознаванию рукописных цифр
- При тонкой настройке для классификации у нас есть определённое количество меток классов (например, «спам» и «не спам»), которые модель может выдавать на выходе
- Модель, дообученная для классификации, может предсказывать только те классы, которые она видела во время обучения (например, «спам» или «не спам»), в то время как модель с инструктивной тонкой настройкой обычно способна выполнять множество задач
- Модель, дообученную для классификации, можно рассматривать как очень узкоспециализированную модель; на практике создать специализированную модель гораздо проще, чем модель-универсала, хорошо справляющуюся со множеством различных задач

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/03.webp" width=800px>

&nbsp;
### 6.2. Подготовка данных

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/04.webp" width=800px>

- В этом разделе мы подготавливаем набор данных, который будем использовать для тонкой настройки классификации.
- Мы используем набор данных, состоящий из спам- и не спам-сообщений, чтобы дообучить LLM классифицировать их.
- Сначала мы скачиваем и распаковываем набор данных.

In [3]:
import requests
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} уже существует. Пропускаем скачивание и распаковку.")
        return

    # Скачивание файла
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(zip_path, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    # Распаковка файла
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Добавляем расширение .tsv
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"Файл скачан и сохранён как {data_file_path}")


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Основной URL не сработал: {e}. Пробуем резервный URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)



# Изначально в книге использовался код ниже.
# Однако urllib использует старые настройки протокола,
# что может вызывать проблемы у некоторых читателей, использующих VPN.
# Версия с `requests` более надёжна в этом отношении.

"""
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} уже существует. Пропускаем скачивание и распаковку.")
        return

    # Скачивание файла
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Распаковка файла
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Добавляем расширение .tsv
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"Файл скачан и сохранён как {data_file_path}")

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
    print(f"Основной URL не сработал: {e}. Пробуем резервный URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
"""

Файл скачан и сохранён как sms_spam_collection\SMSSpamCollection.tsv


'\nimport urllib.request\nimport zipfile\nimport os\nfrom pathlib import Path\n\nurl = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"\nzip_path = "sms_spam_collection.zip"\nextracted_path = "sms_spam_collection"\ndata_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"\n\ndef download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):\n    if data_file_path.exists():\n        print(f"{data_file_path} уже существует. Пропускаем скачивание и распаковку.")\n        return\n\n    # Скачивание файла\n    with urllib.request.urlopen(url) as response:\n        with open(zip_path, "wb") as out_file:\n            out_file.write(response.read())\n\n    # Распаковка файла\n    with zipfile.ZipFile(zip_path, "r") as zip_ref:\n        zip_ref.extractall(extracted_path)\n\n    # Добавляем расширение .tsv\n    original_file_path = Path(extracted_path) / "SMSSpamCollection"\n    os.rename(original_file_path, data_file_path)\n    print(f"Файл скачан и со

- Набор данных сохранён в виде текстового файла с табуляцией в качестве разделителя, который мы можем загрузить в DataFrame библиотеки pandas

In [4]:
import pandas as pd

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


- Когда мы проверяем распределение классов, мы видим, что данные содержат "ham" (то есть «не спам») гораздо чаще, чем "spam"

In [5]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


- Для простоты, а также потому, что для учебных целей мы в любом случае предпочитаем небольшой набор данных (это позволит быстрее выполнить тонкую настройку LLM), мы делаем подвыборку (недостаточную выборку) набора данных так, чтобы он содержал по 747 экземпляров каждого класса

In [6]:
def create_balanced_dataset(df):
    
    # Подсчитываем количество экземпляров "spam"
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # Случайным образом выбираем экземпляры "ham" в количестве, равном числу "spam"
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # Объединяем подвыборку "ham" со "spam"
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


- Далее мы заменяем строковые метки классов "ham" и "spam" на целочисленные метки 0 и 1:

In [7]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})    

In [8]:
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


- Теперь давайте определим функцию, которая случайным образом разделяет набор данных на обучающую, валидационную и тестовую подвыборки

In [9]:
def random_split(df, train_frac, validation_frac):
    # Перемешиваем весь DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Вычисляем индексы разбиения
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Разбиваем DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# Размер тестовой выборки подразумевается равным 0.2 как остаток

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)